# IMAGINAR*IA* — Experiencias narrativas con IA

Notebook interactivo equivalente a la aplicación web.  
Ejecuta las celdas **en orden**: primero instala dependencias, luego configura, y finalmente lanza el juego.

> **Importante**: es necesario tener una clave para la API de Google Gemini → [aistudio.google.com/apikey](https://aistudio.google.com/apikey)  
> **Imágenes**: Pollinations.AI — gratuito.

In [14]:
# Instalación de la librería para usar la API de Gemini
%pip install google-generativeai ipywidgets requests Pillow --quiet

Note: you may need to restart the kernel to use updated packages.


In [15]:
# Importación de librerías
import json, re, random, time, threading, textwrap, io
from urllib.parse import quote

import requests
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Image as IPImage
import google.generativeai as genai

In [16]:
# VARIABLES LOCALES PARA LA AVENTURA
# Rellena aquí antes de ejecutar el juego

GEMINI_API_KEY = "AIza..."  # ← Tu clave de Gemini (OBLIGATORIO)

# ------------------------------------------------------------------------------
# Tipo de experiencia
# Opciones: 'rol', 'teatro', 'fiesta', 'murder', 'escape', 'custom'
ACTIVITY = 'rol'
CUSTOM_ACTIVITY_DESC = ""  # Solo si ACTIVITY = 'custom'
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Jugadores 
PLAYERS = ["Ana", "Carlos", "Marta"]   # Entre 2 y 6 nombres
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Historia
SETTING = "Venecia en el siglo XVIII"  # Ambientación / temática
PLOT    = "Un valioso manuscrito ha desaparecido durante el Carnaval."  # Opcional
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Duración (turnos base antes de ajustar a nº jugadores)
DURATION_TURNS = 3   # Aprox: 5=corto, 8=medio, 14=largo, 20=épico
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Filtros de contenido
AGE     = 'all'    # 'all', 'teen', 'adult'
FILTERS = {'violence': False, 'romance': False, 'horror': False, 'dark': False}
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Imágenes
GEN_COVER  = True   # Imagen de portada al inicio
GEN_SCENES = True   # Imagen al final de cada turno
# ------------------------------------------------------------------------------
print("✓ Configuración guardada")

✓ Configuración guardada


In [17]:
# FUNCIONES PRINCIPALES

# ------------------------------------------------------------------------------
# Prompts que se enviarán a la API

# Filtros
def content_rules(age, filters):
    rules = []
    if age == 'all':  rules.append('Contenido apto para todos los públicos, sin violencia, lenguaje soez ni temas adultos.')
    if age == 'teen': rules.append('Contenido apto para mayores de 12. Puedes incluir tensión dramática moderada.')
    if age == 'adult': rules.append('Contenido para adultos.')
    if not filters.get('violence'): rules.append('Evita descripciones de violencia explícita.')
    if not filters.get('romance'):  rules.append('Evita contenido romántico o sexual.')
    if not filters.get('horror'):   rules.append('Evita elementos de terror o gore.')
    if not filters.get('dark'):     rules.append('Mantén un tono positivo, evita humor negro o temas oscuros.')
    rules.append('REGLA DE REALISMO: Los jugadores solo pueden usar objetos de su inventario. Si intentan algo imposible, falla narrativamente.')
    return ' '.join(rules)

# JSON de respuesta, formato que devolverá la API
JSON_SCHEMA = '''
Responde SIEMPRE con este JSON exacto (sin texto fuera del JSON):
{{
  "narration": "...",
  "question": "...",
  "reward_player": null,
  "reward_item": null,
  "reward_desc": null,
  "special_event": false,
  "is_final": false,
  "image_prompt": "Descripción en inglés (max 20 palabras) de la escena para generar imagen. OBLIGATORIO."
}}
reward_item debe ser un {item_type} o null.
'''

# Creamos un JSON dividido en varios campos, cada uno para cada experiencia de todas las posibles, y donde
# aparecen todos los prompst para que la respuesta sea la esperada
ACTIVITIES = {
    'rol': {
        'name': 'Partida de rol', 'icon': '⚔️', 'item_label': 'objeto mágico o arma',
        'end_title': '⚔ Fin de la aventura',
        'system_fn': lambda G: (
            f"Eres un Dungeon Master narrativo para {len(G['players'])} jugadores: {', '.join(G['players'])}.\n"
            f"Ambientación: {G['setting']}. Trama: {G['plot'] or 'Una aventura épica llena de peligros.'}.\n"
            f"Duración: ~{G['total_turns']} turnos. {content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Narración rica y profunda. Cada turno presenta un dilema narrativo. "
            f"Otorga objeto SOLO ante éxito crítico (~15% turnos). reward_item: null la mayoría de veces.\n"
            + JSON_SCHEMA.format(item_type='objeto mágico o arma')
        )
    },
    'teatro': {
        'name': 'Obra de teatro', 'icon': '🎭', 'item_label': 'accesorio de atrezzo',
        'end_title': '🎭 Telón final',
        'system_fn': lambda G: (
            f"Eres el director de una obra improvisada para {len(G['players'])} actores: {', '.join(G['players'])}.\n"
            f"Temática: {G['setting']}. Premisa: {G['plot'] or 'Una historia dramática llena de giros.'}.\n"
            f"La obra tiene ~{G['total_turns']} escenas. {content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Describe la escena y pide al actor que improvise. "
            f"Accesorio solo si la actuación es magistral (raramente).\n"
            + JSON_SCHEMA.format(item_type='accesorio de atrezzo')
        )
    },
    'fiesta': {
        'name': 'Fiesta temática', 'icon': '🎉', 'item_label': 'punto / premio',
        'end_title': '🎉 ¡La fiesta termina!',
        'system_fn': lambda G: (
            f"Eres el animador de una fiesta para {len(G['players'])} participantes: {', '.join(G['players'])}.\n"
            f"Temática: {G['setting']}. {content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Propón minijuegos físicos reales: mímica, trabalenguas, retos de acción, trivia temática, "
            f"karaoke relámpago, desafíos creativos. Tono de concurso televisivo ágil. "
            f"Punto SOLO si la resolución es muy divertida.\n"
            + JSON_SCHEMA.format(item_type='punto o premio especial')
        )
    },
    'murder': {
        'name': 'Murder mystery', 'icon': '🔍', 'item_label': 'pista clave o evidencia',
        'end_title': '🔍 El caso se cierra',
        'system_fn': lambda G: (
            f"Eres el narrador de un misterio para {len(G['players'])} investigadores: {', '.join(G['players'])}.\n"
            f"Escenario: {G['setting']}. Caso: {G['plot'] or 'Un crimen inexplicable.'}.\n"
            f"{content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Distribuye pistas falsas y verdaderas. Pista real solo ante deducción brillante (~20%).\n"
            + JSON_SCHEMA.format(item_type='pista clave o evidencia')
        )
    },
    'escape': {
        'name': 'Escape room', 'icon': '🚪', 'item_label': 'herramienta o código',
        'end_title': '🚪 ¡Escape completado!',
        'system_fn': lambda G: (
            f"Eres el narrador de un escape room para {len(G['players'])} jugadores: {', '.join(G['players'])}.\n"
            f"Sala: {G['setting']}. {G['plot'] or ''}\n"
            f"{content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Cada turno un puzzle. Herramienta solo si lo resuelve lógicamente.\n"
            + JSON_SCHEMA.format(item_type='herramienta o código')
        )
    },
    'custom': {
        'name': 'Personalizado', 'icon': '✨', 'item_label': 'recompensa temática',
        'end_title': '✨ Fin de la experiencia',
        'system_fn': lambda G: (
            f"Eres el narrador de una experiencia para {len(G['players'])} participantes: {', '.join(G['players'])}.\n"
            f"Tipo: {G.get('custom_activity', '')}. Ambientación: {G['setting']}.\n"
            f"{content_rules(G['age'], G['filters'])}\n"
            f"MECÁNICAS: Recompensa solo ante acciones verdaderamente excepcionales.\n"
            + JSON_SCHEMA.format(item_type='recompensa temática')
        )
    }
}

# ------------------------------------------------------------------------------
# Cofiguración para la API de Gemini

def build_gemini_client(api_key):
    genai.configure(api_key=api_key)
    return genai.GenerativeModel(
        model_name='gemini-2.0-flash',
        generation_config=genai.GenerationConfig(
            response_mime_type='application/json',
            max_output_tokens=1000
        )
    )

def call_gemini(model, system_prompt, history, user_message):
    """Llama a Gemini con historial de conversación y devuelve el texto de respuesta."""
    chat = model.start_chat(history=history)
    response = chat.send_message(
        user_message,
        generation_config=genai.GenerationConfig(
            response_mime_type='application/json',
            max_output_tokens=1000,
            system_instruction=system_prompt  # workaround para system en chat
        )
    )
    return response.text

def call_gemini_once(model, prompt, max_tokens=500):
    """Llamada única sin historial."""
    m = genai.GenerativeModel(
        model_name='gemini-2.0-flash',
        generation_config=genai.GenerationConfig(max_output_tokens=max_tokens)
    )
    return m.generate_content(prompt).text

def parse_json(raw):
    """Extrae el primer objeto JSON de un string."""
    try:
        m = re.search(r'\{[\s\S]*\}', raw)
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Configuración para la API de Pollinations

def pollinations_url(prompt, style='cinematic fantasy illustration, dark atmospheric', width=800, height=450):
    full = f"{style}: {prompt}. No text or letters in image."
    seed = random.randint(0, 999999)
    encoded = quote(full[:500])
    return f"https://image.pollinations.ai/prompt/{encoded}?width={width}&height={height}&seed={seed}&model=flux&referrer=imaginaria"

def fetch_image(url, timeout=90):
    """Descarga la imagen y devuelve bytes, o None si falla."""
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and 'image' in r.headers.get('Content-Type',''):
            return r.content
    except Exception:
        pass
    return None
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Estado del juego

class GameState:
    def __init__(self):
        self.reset()

    def reset(self):
        self.players     = []
        self.setting     = ''
        self.plot        = ''
        self.activity_id = 'rol'
        self.custom_activity = ''
        self.age         = 'all'
        self.filters     = {'violence': False, 'romance': False, 'horror': False, 'dark': False}
        self.total_turns = 8
        self.turn        = 1
        self.player_idx  = 0
        self.inventory   = {}
        self.context     = []   # historial Gemini [{role, parts}]
        self.log         = []
        self.finished    = False
        self.model       = None
        self.gen_scenes  = True

    def to_dict(self):
        return {
            'players': self.players, 'setting': self.setting,
            'plot': self.plot, 'activity_id': self.activity_id,
            'custom_activity': self.custom_activity,
            'age': self.age, 'filters': self.filters,
            'total_turns': self.total_turns
        }

    @property
    def current_player(self):
        return self.players[self.player_idx] if self.players else '?'

    @property
    def activity(self):
        return ACTIVITIES.get(self.activity_id, ACTIVITIES['rol'])

    @property
    def system_prompt(self):
        return self.activity['system_fn'](self.to_dict())

    def add_to_context(self, role, text):
        self.context.append({'role': role, 'parts': [{'text': text}]})
        if len(self.context) > 24:
            self.context = self.context[-24:]
            while self.context and self.context[0]['role'] != 'user':
                self.context.pop(0)

    def add_inventory(self, player, item, desc=''):
        if player not in self.inventory:
            self.inventory[player] = []
        self.inventory[player].append(item)
        self.log.append(f"[Recompensa] {player}: {item}{' — ' + desc if desc else ''}")

    def inventory_str(self):
        result = {p: v for p, v in self.inventory.items() if v}
        return json.dumps(result, ensure_ascii=False) if result else '{}'

G = GameState()

In [18]:
# INTERFAZ INTERACTIVA

# Estilos
CSS = """
<style>
.imag-card {
    background: #201c2e; border: 1px solid rgba(200,170,255,0.2);
    border-radius: 12px; padding: 18px 20px; margin-bottom: 12px;
    font-family: 'Georgia', serif;
}
.imag-title {
    font-family: 'Trebuchet MS', sans-serif; font-size: 10px;
    font-weight: 700; letter-spacing: 2.5px; text-transform: uppercase;
    color: #c9a84c; margin-bottom: 10px;
}
.imag-narrative {
    background: #13111a; border-left: 3px solid #c9a84c;
    border-radius: 0 8px 8px 0; padding: 14px 18px;
    font-style: italic; font-size: 16px; line-height: 1.8;
    color: #ede6f8; white-space: pre-wrap; margin-bottom: 10px;
}
.imag-question {
    background: rgba(124,92,191,0.1); border: 1px solid rgba(124,92,191,0.3);
    border-radius: 8px; padding: 10px 14px; margin-bottom: 10px;
    font-size: 15px; color: #ede6f8;
}
.imag-turn-badge {
    display: inline-block; background: #2a2440;
    border: 1px solid rgba(200,170,255,0.2); border-radius: 999px;
    padding: 5px 16px; font-family: 'Trebuchet MS', sans-serif;
    font-size: 12px; font-weight: 600; color: #ede6f8; margin-bottom: 10px;
}
.imag-progress-wrap {
    background: #13111a; border-radius: 999px; height: 4px;
    margin-bottom: 14px; overflow: hidden;
}
.imag-progress-fill {
    height: 100%; background: linear-gradient(90deg, #7c5cbf, #c9a84c);
    border-radius: 999px; transition: width 0.4s ease;
}
.imag-inv-player {
    display: inline-block; background: #13111a;
    border: 1px solid rgba(200,170,255,0.15);
    border-radius: 8px; padding: 8px 12px; margin: 4px;
    font-size: 13px; vertical-align: top; min-width: 140px;
}
.imag-inv-name { color: #c9a84c; font-weight: 700; font-size: 11px;
    font-family: 'Trebuchet MS', sans-serif; letter-spacing: 0.5px;
    margin-bottom: 4px; }
.imag-item { display: inline-block; background: rgba(201,168,76,0.1);
    border: 1px solid rgba(201,168,76,0.25); border-radius: 4px;
    padding: 1px 7px; font-size: 11px; color: #e8c96a;
    margin: 2px 2px 0 0; }
.imag-reward { background: rgba(39,174,96,0.1);
    border: 1px solid rgba(39,174,96,0.3); border-radius: 8px;
    padding: 8px 14px; color: #7be0a0; font-size: 14px; margin-bottom: 10px; }
.imag-log-entry { font-size: 12px; color: #5a4878;
    padding: 3px 0; border-bottom: 1px solid rgba(200,170,255,0.08); }
.imag-header { font-family: 'Trebuchet MS', sans-serif;
    font-size: 28px; font-weight: 300; letter-spacing: 8px;
    color: #ede6f8; text-align: center; margin-bottom: 4px; }
.imag-header span { color: #e8c96a; font-weight: 700; }
.imag-sub { font-style: italic; color: #9a8ab8; font-size: 12px;
    text-align: center; letter-spacing: 1.5px; margin-bottom: 20px; }
.imag-end-title { font-family: 'Georgia', serif; font-size: 24px;
    color: #e8c96a; text-align: center; letter-spacing: 3px;
    margin-bottom: 16px; }
.imag-img { width: 100%; border-radius: 10px;
    border: 1px solid rgba(200,170,255,0.2); margin-bottom: 12px; }
</style>
"""

# ─── Helpers de HTML ──────────────────────────────────────────────────────────

def h(tag, content='', **attrs):
    attr_str = ' '.join(f'{k.replace("_","-")}="{v}"' for k,v in attrs.items())
    return f'<{tag} {attr_str}>{content}</{tag}>'

def render_narrative(narration, question=''):
    q = f'<div class="imag-question">{question}</div>' if question else ''
    return f'<div class="imag-narrative">{narration}</div>{q}'

def render_turn_header(G):
    round_n  = (G.turn - 1) // len(G.players) + 1
    rounds_t = (G.total_turns - 1) // len(G.players) + 1
    pct = max(0, min(100, int(((G.turn - 1) / G.total_turns) * 100)))
    act = G.activity
    return (
        f'<div class="imag-turn-badge">{act["icon"]} {act["name"]} '
        f'· Ronda {round_n}/{rounds_t} · Turno {G.turn}/{G.total_turns} '
        f'· 🎯 {G.current_player}</div>'
        f'<div class="imag-progress-wrap">'
        f'<div class="imag-progress-fill" style="width:{pct}%"></div></div>'
    )

def render_inventory(G):
    if not any(G.inventory.values()):
        return '<div style="color:#5a4878;font-style:italic;font-size:13px">Sin objetos todavía.</div>'
    items_html = ''
    for p, items in G.inventory.items():
        tags = ''.join(f'<span class="imag-item">{i}</span>' for i in items) if items else '<span style="color:#5a4878;font-size:12px">—</span>'
        items_html += f'<div class="imag-inv-player"><div class="imag-inv-name">{p}</div>{tags}</div>'
    return items_html

def render_log(G, max_entries=8):
    entries = G.log[-max_entries:]
    return ''.join(f'<div class="imag-log-entry">{e}</div>' for e in reversed(entries)) or \
           '<div style="color:#5a4878;font-size:12px;font-style:italic">El registro está vacío.</div>'

# ─── Widgets ──────────────────────────────────────────────────────────────────

out_header   = widgets.Output()
out_cover    = widgets.Output()
out_narrative= widgets.Output()
out_image    = widgets.Output()
out_inventory= widgets.Output()
out_log      = widgets.Output()
out_reward   = widgets.Output()
out_status   = widgets.Output()   # mensajes de carga / error

txt_response = widgets.Textarea(
    placeholder='Describe qué hace o dice tu personaje...',
    layout=widgets.Layout(width='100%', height='80px')
)
btn_send = widgets.Button(
    description='Enviar →',
    button_style='warning',
    layout=widgets.Layout(width='130px')
)
btn_end = widgets.Button(
    description='Terminar',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

input_area = widgets.VBox([
    widgets.HTML('<div style="font-size:12px;color:#9a8ab8;margin-bottom:4px">Tu respuesta:</div>'),
    txt_response,
    widgets.HBox([btn_end, btn_send])
])

# ─── Funciones de actualización de UI ────────────────────────────────────────

def refresh_header():
    with out_header:
        clear_output(wait=True)
        display(HTML(render_turn_header(G)))

def refresh_inventory():
    with out_inventory:
        clear_output(wait=True)
        display(HTML(
            '<div class="imag-title">🎒 Inventario / Pistas</div>' +
            render_inventory(G)
        ))

def refresh_log():
    with out_log:
        clear_output(wait=True)
        display(HTML(
            '<div class="imag-title">📋 Registro</div>' +
            render_log(G)
        ))

def show_status(msg, color='#9a8ab8'):
    with out_status:
        clear_output(wait=True)
        display(HTML(f'<div style="font-size:13px;color:{color};font-style:italic">{msg}</div>'))

def clear_status():
    with out_status:
        clear_output(wait=True)

def show_narrative(narration, question=''):
    with out_narrative:
        clear_output(wait=True)
        display(HTML(render_narrative(narration, question)))

def show_reward(msg):
    with out_reward:
        clear_output(wait=True)
        display(HTML(f'<div class="imag-reward">✦ {msg}</div>'))

def clear_reward():
    with out_reward:
        clear_output(wait=True)

def show_scene_image(image_prompt, activity_name):
    """Genera y muestra imagen de escena en un hilo separado para no bloquear."""
    if not G.gen_scenes or not image_prompt:
        return
    def _fetch():
        url  = pollinations_url(image_prompt, f'{activity_name} scene illustration, dark atmospheric')
        data = fetch_image(url)
        if data:
            with out_image:
                clear_output(wait=True)
                display(IPImage(data=data, width=700))
    threading.Thread(target=_fetch, daemon=True).start()

# ─── Lógica principal del turno ───────────────────────────────────────────────

def process_dm_response(raw):
    """Procesa la respuesta JSON del DM y actualiza el estado."""
    p = parse_json(raw)
    if not p:
        show_narrative(raw)
        return p

    narration = p.get('narration', '')
    question  = p.get('question', '')
    show_narrative(narration, question)

    if p.get('reward_item'):
        player = p.get('reward_player') or G.current_player
        item   = p['reward_item']
        desc   = p.get('reward_desc', '')
        G.add_inventory(player, item, desc)
        show_reward(f"{player} obtiene: {item}" + (f" — {desc}" if desc else ''))
        refresh_inventory()

    if p.get('special_event'):
        show_reward('¡Evento especial!')

    if p.get('image_prompt'):
        show_scene_image(p['image_prompt'], G.activity['name'])

    if p.get('is_final') or G.turn >= G.total_turns:
        G.finished = True
        time.sleep(1.2)
        finish_game(narration)

    return p

def load_situation():
    """Carga la situación del turno actual (sin respuesta del jugador)."""
    cp      = G.current_player
    is_final= G.turn >= G.total_turns
    inv_str = G.inventory_str()

    msg = (f"Turno {G.turn}/{G.total_turns}."
           + (" TURNO FINAL — is_final: true." if is_final else "")
           + f" Participante activo: {cp}."
           + (f" Inventario: {inv_str}." if inv_str != '{}' else ""))

    show_status('⏳ Preparando escena...')
    try:
        G.add_to_context('user', msg)
        # Llamada directa a Gemini con historial
        chat = G.model.start_chat(history=G.context[:-1])  # sin el último que acabamos de añadir
        resp = chat.send_message(msg)
        raw  = resp.text
        G.add_to_context('model', raw)
        clear_status()
        process_dm_response(raw)
    except Exception as e:
        show_status(f'❌ Error: {e}', color='#e07060')

def submit_response(b):
    """Callback del botón Enviar."""
    if G.finished:
        return
    resp = txt_response.value.strip()
    if not resp:
        show_status('⚠ Escribe tu respuesta antes de enviar.', '#e8c96a')
        return

    btn_send.disabled = True
    btn_end.disabled  = True
    clear_reward()
    txt_response.value = ''

    cp      = G.current_player
    is_final= G.turn >= G.total_turns
    next_p  = G.players[(G.player_idx + 1) % len(G.players)]
    G.log.append(f'{cp}: "{resp}"')
    refresh_log()

    eval_msg = (
        f'{cp} responde: "{resp}". ' +
        ('RESPUESTA FINAL. Concluye con un epílogo memorable. is_final: true.'
         if is_final
         else f'Evalúa, otorga recompensa si merece, y plantea la siguiente situación a: {next_p}.')
    )

    show_status('⏳ Evaluando respuesta...')
    try:
        G.add_to_context('user', eval_msg)
        chat = G.model.start_chat(history=G.context[:-1])
        resp_raw = chat.send_message(eval_msg).text
        G.add_to_context('model', resp_raw)
        clear_status()
        p = process_dm_response(resp_raw)

        if not G.finished:
            G.player_idx = (G.player_idx + 1) % len(G.players)
            G.turn += 1
            refresh_header()
            refresh_log()

    except Exception as e:
        show_status(f'❌ Error: {e}', '#e07060')
    finally:
        btn_send.disabled = False
        btn_end.disabled  = False

def finish_game(final_narr=''):
    """Muestra la pantalla de fin."""
    act = G.activity
    inv_html = ''
    for p, items in G.inventory.items():
        tags = ''.join(f'<span class="imag-item">{i}</span>' for i in items) or '<span style="color:#5a4878">Sin objetos</span>'
        inv_html += f'<div class="imag-inv-player"><div class="imag-inv-name">{p}</div>{tags}</div>'

    end_html = (
        f'<div class="imag-end-title">{act["end_title"]}</div>'
        f'<div class="imag-narrative">{final_narr or "La experiencia ha llegado a su fin."}</div>'
        f'<div class="imag-title" style="margin-top:16px">🎒 Objetos y pistas recogidos</div>'
        f'{inv_html}'
    )
    with out_narrative:
        clear_output(wait=True)
        display(HTML(end_html))
    input_area.layout.display = 'none'
    show_status('✓ Experiencia completada. Ejecuta la celda de nuevo para una nueva partida.', '#27ae60')

def end_game_click(b):
    G.finished = True
    finish_game()

btn_send.on_click(submit_response)
btn_end.on_click(end_game_click)

print("✓ Interfaz lista")

✓ Interfaz lista


In [19]:
# BUCLE PRINCIPAL

# Función principal que pone en marcha todo el juego

import math

# Inicializar estado con la configuración ya especificada
G.reset()
G.players     = PLAYERS
G.setting     = SETTING
G.plot        = PLOT
G.activity_id = ACTIVITY
G.custom_activity = CUSTOM_ACTIVITY_DESC
G.age         = AGE
G.filters     = FILTERS.copy()
G.gen_scenes  = GEN_SCENES
G.total_turns = math.ceil(DURATION_TURNS / len(G.players)) * len(G.players)
for p in G.players:
    G.inventory[p] = []

# Configurar Gemini con system prompt como primer mensaje de sistema
genai.configure(api_key=GEMINI_API_KEY)
G.model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    system_instruction=G.system_prompt,
    generation_config=genai.GenerationConfig(
        response_mime_type='application/json',
        max_output_tokens=1000
    )
)

act = G.activity
input_area.layout.display = ''
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Mostrar cabecera
display(HTML(CSS))
display(HTML(
    f'<div class="imag-header">IMAGINAR<span>IA</span></div>'
    f'<div class="imag-sub">experiencias narrativas colaborativas con inteligencia artificial</div>'
))
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Portada (Pollinations)
if GEN_COVER:
    print('🎨 Generando imagen de portada... (puede tardar ~30 s)')
    cover_url  = pollinations_url(
        f"{act['name']} scene: {SETTING}. {PLOT}",
        'epic cinematic wide shot illustration, rich colors, dramatic lighting'
    )
    cover_data = fetch_image(cover_url)
    if cover_data:
        display(IPImage(data=cover_data, width=700))
    else:
        print('⚠ No se pudo cargar la imagen de portada.')
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Equipo inicial
print('⚙ Generando equipo inicial...')
equip_prompt = (
    f"Genera el equipo inicial para: {', '.join(G.players)}. Ambientación: '{SETTING}'.\n"
    f"Asigna a CADA jugador EXACTAMENTE 3 objetos (arma básica, posesión personal, objeto de supervivencia).\n"
    f"Responde ÚNICAMENTE con JSON válido: {{\"Nombre\": [\"obj1\", \"obj2\", \"obj3\"], ...}}"
)
equip_model = genai.GenerativeModel('gemini-2.0-flash',
    generation_config=genai.GenerationConfig(max_output_tokens=500))
try:
    equip_raw = equip_model.generate_content(equip_prompt).text
    equip_data = parse_json(equip_raw)
    if equip_data:
        for p in G.players:
            if p in equip_data and isinstance(equip_data[p], list):
                G.inventory[p] = equip_data[p]
except Exception:
    pass  # Si falla, jugadores sin equipo inicial
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Introducción
print('📖 Generando introducción...')
intro_msg = (
    f"Comienza la experiencia con una introducción vibrante (3-4 oraciones) que establezca el escenario. "
    f"Luego plantea la primera situación al participante: {G.players[0]}. Solo JSON."
)
G.add_to_context('user', intro_msg)
chat     = G.model.start_chat(history=[])
intro_r  = chat.send_message(intro_msg)
intro_raw= intro_r.text
G.add_to_context('model', intro_raw)
intro_p  = parse_json(intro_raw)

clear_output(wait=True)
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Montar la interfaz completa
display(HTML(CSS))
display(HTML(
    f'<div class="imag-header">IMAGINAR<span>IA</span></div>'
    f'<div class="imag-sub">experiencias narrativas colaborativas con inteligencia artificial</div>'
))

if GEN_COVER and cover_data:
    display(IPImage(data=cover_data, width=700))

# Secciones
with out_header:    clear_output(); display(HTML(render_turn_header(G)))
with out_image:     clear_output()
with out_reward:    clear_output()
with out_status:    clear_output()

if intro_p:
    show_narrative(intro_p.get('narration',''), intro_p.get('question',''))
    if intro_p.get('reward_item'):
        G.add_inventory(G.players[0], intro_p['reward_item'], intro_p.get('reward_desc',''))
else:
    show_narrative(intro_raw)

refresh_inventory()
refresh_log()

display(HTML('<div class="imag-card">'),
        out_header,
        out_image,
        out_narrative,
        out_reward,
        input_area,
        out_status,
        HTML('</div>'),
        HTML('<div class="imag-card">'),
        out_inventory,
        HTML('</div>'),
        HTML('<div class="imag-card">'),
        out_log,
        HTML('</div>')
)

🎨 Generando imagen de portada... (puede tardar ~30 s)
⚠ No se pudo cargar la imagen de portada.
⚙ Generando equipo inicial...
📖 Generando introducción...


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 15.310980967s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-Fre